A Python notebook to verify NER schema used in other datasets (BioDivNER, ClimateIE and IBM CC NER).

## Data and Libraires

In [1]:
from dataset_processing import biodivner_process_bio_documents, BIODIVNER_LABELS
from dataset_processing import cwed4eta_process_json_file

In [8]:
def check_for_overlaps(data):
    """
    Checks a list of documents for overlapping NER entities.
    Returns a list of error reports.
    """
    errors = []
    is_flat = True

    for doc_id, entry in enumerate(data):
        text = entry.get('text', "")
        entities = entry.get('entities', [])

        # 1. Sort entities by start position
        # We also sort by 'end' as a secondary key just to be deterministic
        sorted_entities = sorted(entities, key=lambda x: (x['start'], x['end']))

        # 2. Iterate and compare with previous
        for i in range(1, len(sorted_entities)):
            prev_ent = sorted_entities[i-1]
            curr_ent = sorted_entities[i]

            # CHECK: Does the current entity start before the previous one ends?
            if curr_ent['start'] < prev_ent['end']:
                is_flat = False
                errors.append({
                    "doc_index": doc_id,
                    "text_snippet": text[:50] + "..." if len(text) > 50 else text,
                    "conflict": (prev_ent, curr_ent),
                    "reason": f"Entity '{curr_ent['text']}' starts at {curr_ent['start']}, "
                              f"but previous entity '{prev_ent['text']}' ends at {prev_ent['end']}."
                })

    return is_flat, errors

def check_for_overlaps_print(data):
    is_flat, errors = check_for_overlaps(data)

    if is_flat:
        print("✅ Success! The data is a valid Flat NER setup.")
    else:
        print(f"❌ Found {len(errors)} overlaps. This is NOT a flat NER setup.\n")
        for e in errors:
            print(f"Doc #{e['doc_index']} | {e['text_snippet']}")
            print(f"  --> Conflict between: {e['conflict'][0]} AND {e['conflict'][1]}")
            print(f"  --> {e['reason']}\n")

## BioDivNER

In [9]:
data = biodivner_process_bio_documents("DATA/BiodivNER", BIODIVNER_LABELS)
check_for_overlaps_print(data)


Processing file: DATA/BiodivNER/dev.csv...
Processing file: DATA/BiodivNER/train.csv...
✅ Success! The data is a valid Flat NER setup.


## ClimateIE

In [16]:
import json
import glob
import os

# 1. Setup path
folder_path = os.path.join("DATA/ClimateIE/human_corpus")
json_files = glob.glob(os.path.join(folder_path, "*.json"))

formatted_data = []

# 2. Load and transform data
for filepath in json_files:
    with open(filepath, 'r', encoding='utf-8') as f:
        doc_content = json.load(f)

    # Convert the "entities" dictionary (keys are IDs) into a List
    # Map 'begin' -> 'start' and 'substring' -> 'text'
    raw_entities = doc_content.get("entities", {})
    clean_entities = []
    
    # Iterate over values since entities is a dict { "93163": {...}, ... }
    for ent in raw_entities.values():
        clean_entities.append({
            "start": ent["begin"],       
            "end": ent["end"],
            "text": ent["substring"],    
            "label": ent["label"]
        })

    # Add to list using the structure your function expects
    formatted_data.append({
        "text": doc_content.get("text", ""),
        "entities": clean_entities,
        "filename": os.path.basename(filepath) 
    })

check_for_overlaps_print(formatted_data)

✅ Success! The data is a valid Flat NER setup.


## IBMCCNER

In [20]:
from dataset_processing import IBMCCNER_DIR, ibmccner_process_bio_documents, IBMCCNER_LABELS
split = ["train", "validation"]
from datasets import load_dataset


# 1. Load and pre-process the data into a document-wise list of lines
print(f"Loading '{IBMCCNER_DIR}' with splits: {split}")
ds = load_dataset(IBMCCNER_DIR) 
train_documentwise = []
temp_list = []
if type(split) == type(["list"]):
    for sp in split:
        for line in ds[sp]["text"]:
            if line.strip().startswith('-DOCSTART-'):
                if temp_list:
                    train_documentwise.append(temp_list)
                temp_list = []
            temp_list.append(line)
        if temp_list:
            train_documentwise.append(temp_list)
else:
    for line in ds[split]["text"]:
        if line.strip().startswith('-DOCSTART-'):
            if temp_list:
                train_documentwise.append(temp_list)
            temp_list = []
        temp_list.append(line)
    if temp_list:
        train_documentwise.append(temp_list)

print("Converting from BIO tags to char spans.")        
structured_documents_char_spans = ibmccner_process_bio_documents(
    document_list=train_documentwise,
    labels_to_keep=IBMCCNER_LABELS
)

check_for_overlaps_print(structured_documents_char_spans)

Loading 'ibm-research/Climate-Change-NER' with splits: ['train', 'validation']
Converting from BIO tags to char spans.
✅ Success! The data is a valid Flat NER setup.
